# Week 2 — Structured Output Foundations

Covers: type hints (`typing`), Pydantic `BaseModel`/`Field`/validators, and decorators.

NOTE: The Pydantic cells require `pip install pydantic` (see `requirements.txt` for this week).
If Pydantic isn't installed yet, the cell will raise `ModuleNotFoundError` — that's expected
until you install it.

## 1. Type hints

In [ ]:
from typing import List, Dict, Optional, Union

def extract_fields(text: str) -> Dict[str, str]:
    """Type hints document intent; they don't enforce anything by themselves —
    that enforcement is exactly what Pydantic adds in the next section."""
    return {"raw_text": text}

def find_urgency(tags: List[str]) -> Optional[str]:
    for tag in tags:
        if tag in {"low", "medium", "high"}:
            return tag
    return None   # explicit: "not found" is a valid outcome

def normalise_id(value: Union[int, str]) -> str:
    return str(value)

print(extract_fields("patient reports fever"))
print(find_urgency(["fda", "high", "labelling"]))
print(normalise_id(42), normalise_id("42"))

## 2. Pydantic — the validation layer every SmartIntake-style capstone relies on

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class IntakeExtraction(BaseModel):
    query_type: str
    regulation_ref: Literal["FDA", "EMA", "CDSCO", "UNKNOWN"] = "UNKNOWN"
    urgency: Literal["low", "medium", "high"]
    submitting_team: str = Field(min_length=1)

    @field_validator("query_type")
    @classmethod
    def query_type_not_empty(cls, v):
        if not v.strip():
            raise ValueError("query_type cannot be blank")
        return v

# Valid case
good = IntakeExtraction(
    query_type="labelling change",
    regulation_ref="FDA",
    urgency="high",
    submitting_team="Regulatory Affairs",
)
print(good)
print(good.model_dump())   # convert back to a plain dict, e.g. for logging/storage

In [ ]:
# Invalid case — Pydantic raises, it doesn't silently accept bad data
from pydantic import ValidationError

try:
    bad = IntakeExtraction(
        query_type="",
        regulation_ref="MADE_UP_AGENCY",   # not in the allowed Literal set
        urgency="asap",                     # not in the allowed Literal set
        submitting_team="",
    )
except ValidationError as e:
    print("Validation failed as expected:")
    for err in e.errors():
        print(" -", err["loc"], err["msg"])

## 3. Decorators

In [ ]:
import time
from functools import wraps

def timed(func):
    @wraps(func)   # preserves the wrapped function's name/docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"{func.__name__} took {elapsed_ms:.2f} ms")
        return result
    return wrapper

@timed
def slow_extraction(text):
    total = 0
    for _ in range(200_000):
        total += 1
    return {"length": len(text)}

print(slow_extraction("some regulatory text"))

This exact pattern — a decorator wrapping a function to add cross-cutting behaviour
(timing, retries, logging, tool registration) — is what powers:
- `@field_validator` above
- `@tool` in agent-tool frameworks (Week 4)
- `@mcp.tool()` in FastMCP (Week 5)

Same mechanism, different job at each layer.